<a href="https://colab.research.google.com/github/enyachang1119-debug/DS2002/blob/main/Copy_of_2026_10_09_%E2%80%94_Public_API_to_Clean_Table_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [19]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    # TODO: loop up to `tries` times
    # TODO:   request with a timeout; return pd.DataFrame(r.json()['daily']) on 200
    # TODO:   raise on a status that is not in RETRY_STATUSES
    # TODO:   otherwise sleep (increasing each time) and try again
    # TODO: raise if every attempt failed

    # Try the request up to the allowed number of attempts
    for attempt in range(tries):
        try:    # Set a timeout so the request does not wait indefinitely
            r = requests.get(BASE, params=params, timeout=10)
        except (requests.Timeout, requests.ConnectionError):
                # Retry network errors, but raise on the final attempt
            if attempt == tries - 1:
                raise
        else: # Convert the daily weather data into a DataFrame
            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])

              # Raise immediately for non-retryable responses, including 400 and 404
            if r.status_code not in RETRY_STATUSES:
                r.raise_for_status()
                raise RuntimeError(f'Unexpected status: {r.status_code}')

              # Raise if the final attempt still returns a retryable error
            if attempt == tries - 1:
                r.raise_for_status()
        # Wait longer before each retry: 1 second, then 2 seconds
        time.sleep(2 ** attempt)


    # Handle cases where no attempts were made
    raise RuntimeError('All weather requests failed')

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [20]:
# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about
used_fallback = False
pulled_at = None
try:
    wx = get_weather('2026-09-05', '2026-10-03')
    # Record when the API data was retrieved.
    pulled_at = pd.Timestamp.now(tz='UTC')
    wx = wx[wx['time'].isin(['2026-09-05', '2026-10-03'])].copy()
except (requests.Timeout, requests.ConnectionError, requests.HTTPError):
    # Record that the API request failed.
    used_fallback = True


### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [21]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'

if used_fallback: # Use sample data only if the API request failed
    wx = FALLBACK.copy()
# Label the data source so sample data is clearly identified.
wx['source'] = 'fallback sample' if used_fallback else 'open-meteo'
wx

,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [22]:
# TODO
# Rename the API columns to the names used in our table
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})

wx['date'] = pd.to_datetime(wx['date']) # Convert date strings into datetime values for joining
wx = wx[['date', 'temp_max', 'precip_mm', 'source']] # Keep the required columns in the specified order
wx

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [23]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [24]:
# TODO
# Find the dates with the highest and lowest precipitation
wetter = wx.loc[wx['precip_mm'].idxmax()]
drier = wx.loc[wx['precip_mm'].idxmin()]

# Calculate the difference in rainfall
difference = wetter['precip_mm'] - drier['precip_mm']

# Print a readable sentence with the dates and rainfall difference
print(
    f"{wetter['date']:%Y-%m-%d} was wetter than "
    f"{drier['date']:%Y-%m-%d} by {difference:.1f} mm of rain."
)

2026-10-03 was wetter than 2026-09-05 by 12.2 mm of rain.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [25]:
# TODO
# Print the API endpoint, location, and requested date range.
print(f"Endpoint: {BASE}")
print("Coordinates: latitude 38.03, longitude -78.51")
print("Requested date range: 2026-09-05 to 2026-10-03")

# Report the retrieval time and data source.
if used_fallback:
    print("Pulled at: unavailable (API request failed)")
    print("Data source: fallback sample")
else:
    print(f"Pulled at: {pulled_at:%Y-%m-%d %H:%M:%S %Z}")
    print("Data source: open-meteo (real data)")

Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: latitude 38.03, longitude -78.51
Requested date range: 2026-09-05 to 2026-10-03
Pulled at: 2026-10-10 19:47:37 UTC
Data source: open-meteo (real data)


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

_A left merge would keep all 200 days of sales, but the weather values would be missing. The chart might show no points because the missing values are skipped. If missing rainfall is filled with zero, it would look like there was no rain on any day. An inner merge would give an empty result. People might not notice because no error was raised and they did not check for missing values._

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [27]:
# stretch
# Fetch Orlando weather and keep the same two dates.
orlando = get_weather(
    '2026-09-05', '2026-10-03',
    lat=28.54, lon=-81.38
)
orlando = orlando[
    orlando['time'].isin(['2026-09-05', '2026-10-03'])
].copy()

# Match the column names and date type of the first city's table.
orlando = orlando.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})
orlando['date'] = pd.to_datetime(orlando['date'])
orlando['source'] = 'open-meteo'
orlando['city'] = 'Orlando'

# Label the first city while preserving its existing source information.
charlottesville = wx.copy()
charlottesville['city'] = 'Charlottesville'

# Stack both cities into one long-format table.
weather_by_city = pd.concat(
    [charlottesville, orlando],
    ignore_index=True
)
weather_by_city = weather_by_city[
    ['city', 'date', 'temp_max', 'precip_mm', 'source']
]

weather_by_city

,city,date,temp_max,precip_mm,source
0,Charlottesville,2026-09-05,29.4,2.5,open-meteo
1,Charlottesville,2026-10-03,22.0,14.7,open-meteo
2,Orlando,2026-09-05,32.1,3.9,open-meteo
3,Orlando,2026-10-03,30.4,14.5,open-meteo
